In [2]:
import pandas as pd
import altair as alt

In [3]:
# Desactivar el límite máximo de filas de Altair por si el dataset final es grande
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [4]:
# 1. Cargar los datos
df_ruido = pd.read_csv('Ruido_con_NaN.csv')
df_clima = pd.read_csv('Clima - Hoja 1.csv')


In [5]:
# 2. Preparar el dataset de Ruido
df_ruido['Fecha'] = pd.to_datetime(df_ruido['Fecha'])

In [6]:
# 3. Preparar el dataset de Clima
df_clima['Temperatura (°C)'] = pd.to_numeric(df_clima['Temperatura (°C)'].astype(str).str.replace(',', '.'), errors='coerce')
df_clima['Precipitaciones (mm)'] = pd.to_numeric(df_clima['Precipitaciones (mm)'].astype(str).str.replace(',', '.'), errors='coerce')
df_clima['Fecha'] = pd.to_datetime(df_clima['Fecha'], errors='coerce')
df_clima['Hora'] = pd.to_datetime(df_clima['Hora'], errors='coerce').dt.hour

C:\Users\carlo\AppData\Local\Temp\ipykernel_15740\2658992014.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clima['Hora'] = pd.to_datetime(df_clima['Hora'], errors='coerce').dt.hour


In [7]:
# 4. Unir (Merge) ambos datasets
df_final = pd.merge(df_ruido, df_clima, on=['Fecha', 'Hora'], how='left')

In [8]:
# 5. CONTEO DE VALORES NaN
nulos_antes = df_final['Medición (dB)'].isna().sum()
print(f"Total de registros: {len(df_final)}")
print(f"Cantidad de valores NaN (horas sin datos de ruido): {nulos_antes}")

Total de registros: 744
Cantidad de valores NaN (horas sin datos de ruido): 97


In [9]:
# 1. Calcular el promedio global de ruido (ignora automáticamente los NaN al calcular)
media_ruido = df_final['Medición (dB)'].mean()
print(f"El promedio global de ruido del mes es: {media_ruido:.2f} dB")

El promedio global de ruido del mes es: 69.01 dB


In [10]:
# 2. Rellenar (imputar) los valores NaN con esta media
df_final['Medición (dB)'] = df_final['Medición (dB)'].fillna(media_ruido)

In [11]:
# 3. Comprobar que el trabajo se hizo correctamente
nulos_despues = df_final['Medición (dB)'].isna().sum()
print(f"Cantidad de valores NaN después de la imputación: {nulos_despues}")

Cantidad de valores NaN después de la imputación: 0


In [12]:
# Mostrar las primeras filas para verificar
df_final.head()

,Fecha,Hora,Medición (dB),Temperatura (°C),Clima,Precipitaciones (mm)
0,2026-03-01,0,69.005443,12.7,Lluvia moderada a intervalos,0.0
1,2026-03-01,1,70.240000,12.4,Lluvia moderada a intervalos,0.0
2,2026-03-01,2,73.225000,12.3,Lluvia moderada a intervalos,0.0
3,2026-03-01,3,75.103333,12.2,Lluvia moderada a intervalos,0.0
4,2026-03-01,4,73.073333,11.9,Neblina,0.0


In [13]:
df_final.to_csv("df_final.csv", index=False)

In [14]:
import altair as alt
import pandas as pd

# 1. Configuración de estilo
alt.theme.enable('fivethirtyeight')

# 2. Preparación de datos
df_hora_promedio = df_final.groupby('Hora')['Medición (dB)'].mean().reset_index()
orden_franjas = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
df_fondo = pd.DataFrame({
    'inicio': [0, 6, 12, 18],
    'fin': [6, 12, 18, 24],
    'Franja': ['Madrugada', 'Mañana', 'Tarde', 'Noche']
})

# 3. GRÁFICA 1: RITMO ACÚSTICO (Versión Compacta)
fondo_sombras = alt.Chart(df_fondo).mark_rect(opacity=0.1).encode(
    x='inicio:Q',
    x2='fin:Q',
    color=alt.Color('Franja:N', 
                    scale=alt.Scale(domain=orden_franjas, scheme='category10'), 
                    legend=alt.Legend(title="Fase", orient='right', labelFontSize=10))
)

linea_tendencia = alt.Chart(df_hora_promedio).mark_line(
    color='#1a1a1a', strokeWidth=2.5, interpolate='monotone'
).encode(
    x=alt.X('Hora:Q', title=None, scale=alt.Scale(domain=[0, 23])),
    y=alt.Y('Medición (dB):Q', title='dB Promedio', scale=alt.Scale(zero=False, domain=[60, 78]))
)

puntos_detalle = alt.Chart(df_hora_promedio).mark_circle(size=50).encode(
    x='Hora:Q',
    y='Medición (dB):Q',
    color=alt.Color('Medición (dB):Q', scale=alt.Scale(scheme='magma'), legend=None)
)

# Altura reducida a 180
grafico_linea = (fondo_sombras + linea_tendencia + puntos_detalle).properties(
    width=700, height=180, title=alt.TitleParams("Ritmo y Densidad Acústica", fontSize=16)
)

# 4. GRÁFICA 2: MAPA DE CALOR (Versión Compacta)
df_final['Fecha_Etiqueta'] = df_final['Fecha'].dt.strftime('%d-%m') # Fecha más corta

mapa_calor = alt.Chart(df_final).mark_rect().encode(
    x=alt.X('Hora:O', title='Hora (0-23)', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Fecha_Etiqueta:O', title='Día/Mes'),
    color=alt.Color('Medición (dB):Q', title='dB', scale=alt.Scale(scheme='magma')),
    tooltip=['Fecha_Etiqueta', 'Hora', 'Medición (dB)']
).properties(
    width=700, height=280 # Altura reducida a 280
)

# 5. JUNTAR Y CONFIGURAR
dashboard_final = (grafico_linea & mapa_calor).configure_axis(
    labelFontSize=9,
    titleFontSize=11
)

dashboard_final.display()

alt.VConcatChart(...)

In [15]:
import altair as alt

# 1. Configuración de estilo
alt.theme.enable('fivethirtyeight')

# 2. CREACIÓN DEL GRÁFICO DE DISPERSIÓN + REGRESIÓN
base = alt.Chart(df_final).encode(
    x=alt.X('Temperatura (°C):Q', title='Temperatura (°C)', scale=alt.Scale(zero=False)),
    y=alt.Y('Medición (dB):Q', title='Nivel de Ruido (dB)', scale=alt.Scale(zero=False))
)

# Capa 1: Los puntos (con un color suave para que no saturen)
puntos = base.mark_circle(size=60, opacity=0.4).encode(
    color=alt.Color('Momento_Dia:N', title='Momento del Día', scale=alt.Scale(scheme='set1')),
    tooltip=['Fecha', 'Hora', 'Temperatura (°C)', 'Medición (dB)', 'Clima']
)

# Capa 2: La línea de tendencia (Regresión lineal)
# Esto le dice al profe si a más calor hay más ruido o no
linea_tendencia = base.transform_regression('Temperatura (°C)', 'Medición (dB)').mark_line(
    color='#2c3e50', 
    strokeWidth=4
)

# Combinar
grafico_temperatura_ruido = (puntos + linea_tendencia).properties(
    width=850,
    height=500,
    title=alt.TitleParams(
        "¿Influye el Clima en el Ruido?", 
        subtitle="Relación entre Temperatura y Decibelios con línea de tendencia"
    )
).interactive() # ¡Para que puedas hacer zoom y moverte!

grafico_temperatura_ruido.display()

alt.LayerChart(...)

In [16]:
import altair as alt
import pandas as pd

# 1. Configuración de estilo
alt.theme.enable('fivethirtyeight')

# --- CONSTRUCCIÓN DEL GRÁFICO (Strip Plot) ---

base = alt.Chart(df_final).encode(
    y=alt.Y('Clima:N', title=None, sort='-x'), # Ordenamos climas por nivel de ruido
    x=alt.X('Medición (dB):Q', title='Nivel de Ruido (dB)', scale=alt.Scale(zero=False))
)

# Capa 1: Nube de todos los puntos (pequeños y transparentes)
puntos_fondo = base.mark_tick(opacity=0.2, color='gray').encode(
    color=alt.condition(
        'datum["Medición (dB)"] > 70', # Resaltamos los puntos ruidosos
        alt.value('#e74c3c'),
        alt.value('gray')
    )
)

# Capa 2: Barra gruesa que marca el promedio de cada clima
barras_promedio = base.mark_tick(color='#1a1a1a', thickness=4, size=40).encode(
    x='mean(Medición (dB)):Q',
    tooltip=[
        alt.Tooltip('Clima'),
        alt.Tooltip('mean(Medición (dB)):Q', format='.1f', title='Ruido Promedio (dB)')
    ]
)

# Unimos y damos título
grafico_clima_impacto = (puntos_fondo + barras_promedio).properties(
    width=800, height=500,
    title=alt.TitleParams(
        "Impacto del Clima en la Contaminación Acústica", 
        subtitle="Cada marca gris es una hora real; la barra negra es el promedio del clima"
    )
)

grafico_clima_impacto.display()

alt.LayerChart(...)

In [17]:
import altair as alt
import pandas as pd

# 1. Preparación de los datos (Igual que antes)
cols_interes = ['Medición (dB)', 'Temperatura (°C)', 'Hora']
df_corr = df_final[cols_interes].corr().stack().reset_index()
df_corr.columns = ['Variable 1', 'Variable 2', 'Correlación']

# 2. CREACIÓN DEL MAPA DE CALOR CORREGIDO
grafico_corr = alt.Chart(df_corr).mark_rect().encode(
    x=alt.X('Variable 1:N', title=None),
    y=alt.Y('Variable 2:N', title=None),
    color=alt.Color('Correlación:Q', 
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1]), # Nombre corregido a 'redblue'
                    title="Fuerza"),
    tooltip=[
        alt.Tooltip('Variable 1'),
        alt.Tooltip('Variable 2'),
        alt.Tooltip('Correlación', format='.2f')
    ]
).properties(
    width=400,
    height=400,
    title=alt.TitleParams(
        "Matriz de Correlación: ¿Qué influye más?",
        subtitle="Rojo: Relación directa | Azul: Relación inversa"
    )
)

# Texto con los valores para que sea fácil de leer
texto = grafico_corr.mark_text(baseline='middle').encode(
    text=alt.Text('Correlación:Q', format='.2f'),
    color=alt.condition(
        alt.abs(alt.datum.Correlación) > 0.5, # Si es muy fuerte, texto blanco
        alt.value('white'),
        alt.value('black')
    )
)

(grafico_corr + texto).display()

AttributeError: module 'altair' has no attribute 'abs'